# L6c: Iterative Methods for Solving Linear Algebraic Equations

In this lecture, we will develop iterative methods for solving square systems of linear algebraic equations. These methods begin with an initial guess and use repeated corrections to approach a solution, offering an alternative when a direct solution requires too much computation or memory.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> - **Construct an iterative method:** Use a matrix splitting to derive an update from the residual, and distinguish successful convergence from stopping at an iteration limit.
> - **Assess convergence:** Explain the spectral-radius condition, identify when strict diagonal dominance guarantees convergence, and use an error bound to estimate a sufficient iteration count.
> - **Compare specific methods:** Explain how Jacobi, Gauss–Seidel, and successive over-relaxation construct their updates, and how these choices affect the work per iteration and convergence.

We will begin with the steps shared by these methods, derive the correction from a matrix splitting, and examine the conditions under which repeated corrections approach the solution. We will then use this framework to compare the three methods and interpret their behavior in the companion example.
___


## Examples
We will use the following example to compare iterative solutions with a direct solution and examine their computational cost:

> [▶ Fun with Iterative Methods](CHEME-5800-L6c-Example-FunWithIterativeSolvers-Fall-2026.ipynb). How do repeated corrections compare with a direct solution? We solve a randomly generated linear system using a selected iterative method, compare its result with Julia's direct solution, and benchmark runtime and memory use. Changing the selected method allows us to explore Jacobi, Gauss–Seidel, and successive over-relaxation.

___


## General iterative method
Suppose we have a square system of linear equations given by:

$$
\mathbf{A}\mathbf{x}=\mathbf{b},
$$

where $\mathbf{A}\in\mathbb{R}^{n\times n}$ is the system matrix, $\mathbf{x}\in\mathbb{R}^{n}$ contains the $n$ unknowns, and $\mathbf{b}\in\mathbb{R}^{n}$ is the right-hand side vector. We assume that $\mathbf{A}$ is nonsingular, so the system has a unique solution, and seek a sequence of approximations that approaches this solution.

Starting from an initial guess $\mathbf{x}^{(0)}$, we measure how well the current approximation satisfies the equations using the residual vector:

$$
\mathbf{r}^{(k)}=\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)},
$$

where $k$ counts the corrections already made. Each residual component measures the mismatch in one equation; the residual is zero at the exact solution.

To construct a correction, choose a splitting $\mathbf{A}=\mathbf{M}-\mathbf{N}$, where $\mathbf{M}$ is nonsingular and chosen so that systems with $\mathbf{M}$ are inexpensive to solve, and $\mathbf{N}=\mathbf{M}-\mathbf{A}$. A diagonal or triangular $\mathbf{M}$ is a common choice. Keeping this splitting fixed across iterations gives a _stationary iterative method_.

Let's sketch the steps before deriving the correction.

__Initialize__: Choose an initial guess $\mathbf{x}^{(0)}$, an absolute residual tolerance $\epsilon>0$, and a maximum number of corrections $\texttt{maxiter}$. Set the iteration counter $k\gets0$ and $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:

1. Calculate the residual vector $\mathbf{r}^{(k)}\gets\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)}$.
2. Check whether to stop:
   - If $\|\mathbf{r}^{(k)}\|_2<\epsilon$, set $\texttt{converged}\gets\texttt{true}$ and return $\mathbf{x}^{(k)}$ with this status.
   - Otherwise, if $k\geq\texttt{maxiter}$, return $\mathbf{x}^{(k)}$ with $\texttt{converged}=\texttt{false}$ and a warning that the residual tolerance was not met.
3. Calculate the correction $\mathbf{d}^{(k)}$ by solving $\mathbf{M}\mathbf{d}^{(k)}=\mathbf{r}^{(k)}$.
4. Update the approximation: $\mathbf{x}^{(k+1)}\gets\mathbf{x}^{(k)}+\mathbf{d}^{(k)}$.
5. Increment the iteration counter: $k\gets k+1$.

The correction can be written as $\mathbf{d}^{(k)}=\mathbf{M}^{-1}\mathbf{r}^{(k)}$, but computing it requires solving a system with $\mathbf{M}$; we do not need to form an explicit inverse. The choice of $\mathbf{M}$ affects both the work needed for each correction and whether the iteration converges.

A small residual means that the equations are nearly satisfied. How close the approximation is to the true solution also depends on the sensitivity of the system, so a residual tolerance is not automatically a bound on the solution error. Next, we will derive the correction from the matrix splitting and show how the choice of $\mathbf{M}$ connects the specific methods.

___


## Update direction
Where does the correction $\mathbf{d}^{(k)}$ come from? Starting from the original system and substituting the splitting $\mathbf{A}=\mathbf{M}-\mathbf{N}$ gives:

$$
\begin{align*}
\mathbf{A}\mathbf{x}&=\mathbf{b},\\
(\mathbf{M}-\mathbf{N})\mathbf{x}&=\mathbf{b},\\
\mathbf{M}\mathbf{x}&=\mathbf{b}+\mathbf{N}\mathbf{x}.
\end{align*}
$$

To turn this identity into an iteration, we evaluate the right-hand side using the current approximation $\mathbf{x}^{(k)}$ and solve for the next approximation $\mathbf{x}^{(k+1)}$. Since $\mathbf{M}$ is nonsingular, this defines the update as:

$$
\begin{align*}
\mathbf{M}\mathbf{x}^{(k+1)}&=\mathbf{b}+\mathbf{N}\mathbf{x}^{(k)},\\
\mathbf{x}^{(k+1)}&=\mathbf{M}^{-1}\mathbf{b}+\mathbf{M}^{-1}\mathbf{N}\mathbf{x}^{(k)}.
\end{align*}
$$

Now let's express this update as a correction to the current approximation. Using $\mathbf{N}=\mathbf{M}-\mathbf{A}$, with $\mathbf{I}$ denoting the $n\times n$ identity matrix, we obtain:

$$
\mathbf{M}^{-1}\mathbf{N}
=\mathbf{M}^{-1}(\mathbf{M}-\mathbf{A})
=\mathbf{I}-\mathbf{M}^{-1}\mathbf{A}.
$$

Substituting this identity into the update gives:

$$
\begin{align*}
\mathbf{x}^{(k+1)}
&=\mathbf{M}^{-1}\mathbf{b}+(\mathbf{I}-\mathbf{M}^{-1}\mathbf{A})\mathbf{x}^{(k)}\\
&=\mathbf{x}^{(k)}+\mathbf{M}^{-1}\underbrace{(\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)})}_{\text{residual }\mathbf{r}^{(k)}}\\
&=\mathbf{x}^{(k)}+\underbrace{\mathbf{M}^{-1}\mathbf{r}^{(k)}}_{\text{correction }\mathbf{d}^{(k)}}.
\end{align*}
$$

The correction therefore solves $\mathbf{M}\mathbf{d}^{(k)}=\mathbf{r}^{(k)}$, which is the step used in our general algorithm. Different choices of $\mathbf{M}$ produce the Jacobi, Gauss–Seidel, and successive over-relaxation methods; they determine how the residual is converted into a correction.

> __Stationary iteration:__
>
> Define the _iteration matrix_ $\mathbf{G}$ and the constant vector $\mathbf{c}$ by:
>
> $$
> \mathbf{G}=\mathbf{M}^{-1}\mathbf{N},
> \qquad
> \mathbf{c}=\mathbf{M}^{-1}\mathbf{b}.
> $$
>
> The same update can then be written as:
>
> $$
> \mathbf{x}^{(k+1)}=\mathbf{G}\mathbf{x}^{(k)}+\mathbf{c}.
> $$
>
> The matrix and vector remain fixed during the iteration. An exact solution is unchanged by this update because its residual is zero, but this alone does not establish that the iterates approach it.

We next examine how the iteration matrix determines whether approximations starting from an arbitrary initial guess converge to the solution.

___


## Convergence
One common question that arises when using iterative methods is: _When does the method converge?_ In other words, how do we know that the sequence of iterates $\{\mathbf{x}^{(k)}\}$ will approach the true solution $\mathbf{x}^{\star}$ as $k$ increases? A stationary iteration $\mathbf{x}^{(k+1)}=\mathbf{G}\,\mathbf{x}^{(k)}+\mathbf{c}$ converges __for every initial guess__ $\mathbf{x}^{(0)}$ __if and only if__:
$$
\rho(\mathbf{G}) = \rho\bigl(\mathbf{M}^{-1}\mathbf{N}\bigr) \;<\;1.
$$
The spectral radius $\rho(\mathbf{G}) = \;\max_i|\lambda_i|$, where $\lambda_i$ are the eigenvalues of the iteration matrix $\mathbf{G}$. This means that the method will converge to the true solution regardless of the initial guess $\mathbf{x}^{(0)}$.

> __Additional Notes:__ The derivation of the spectral radius convergence condition is provided in the [Advanced: Where does spectral radius convergence condition come from?](CHEME-5800-L6c-Advanced-Convergence-IterativeMethods-Fall-2026.ipynb) notebook. Check it out if you are interested in the mathematical details behind this condition.

### Diagonal dominance

If the coefficient matrix $\mathbf{A}$ is **strictly diagonally dominant**, Jacobi and Gauss–Seidel iterations converge from any initial guess, so diagonal dominance is **sufficient but not necessary** condition for convergence.

Recall that we split the matrix $\mathbf{A}$ into its diagonal and off-diagonal components: $\mathbf{A} = \mathbf{M} - \mathbf{N}$, where $\mathbf{M}$ is the diagonal part of $\mathbf{A}$ and $\mathbf{N}$ is the off-diagonal part. The iteration matrix is then given by: $\mathbf{G} = \mathbf{M}^{-1}\mathbf{N}$. Thus, for a matrix $\mathbf{A}$ to be strictly diagonally dominant, we require that for each row $i$:
$$
    |\mathbf{M}_{ii}| \;>\; \sum_{j\neq i} |\mathbf{N}_{ij}|,
$$
This is related to the spectral radius of the iteration matrix $\mathbf{G}$ through the induced infinity-norm:
$$
    \|\mathbf{G}\|_\infty 
    = \max \left\{\sum_j \bigl| (\mathbf{M}^{-1}\mathbf{N})_{ij}\bigr|\right\}_{i} 
    \;\le\;\max_i \frac{1}{|\mathbf{M}_{ii}|}\sum_{j\neq i}|\mathbf{N}_{ij}|
    \;<\;1.
$$
A basic property of induced norms is that for any eigenpair $(\lambda, \mathbf{v})$ of $\mathbf{G}$ with eigenvector $\mathbf{v}\neq0$:
$$
|\lambda|\;\|\mathbf{v}\|_\infty \;=\;\|\lambda \mathbf{v}\|_\infty
\;=\;\|\mathbf{G}\mathbf{v}\|_\infty
\;\le\;\|\mathbf{G}\|_\infty\,\|\mathbf{v}\|_\infty
\quad\Longrightarrow\quad
|\lambda|\le\|\mathbf{G}\|_\infty.
$$
Taking the maximum over all eigenvalues gives
$\rho(\mathbf{G})=\max_i|\lambda_i|\le\|\mathbf{G}\|_\infty$.


### Rate of convergence?
The rate of convergence of an iterative method is related to the spectral radius of the iteration matrix $\mathbf{G}$. Let's first define the error vector at iteration $k$ as:
$$
\mathbf{e}^{(k)} = \mathbf{x}^{(k)} - \mathbf{x}^{\star}
$$
where $\mathbf{x}^{(k)}$ is the current iterate and $\mathbf{x}^{\star}$ is the true solution. Now, let's consider a _worst-case_ bound. Suppose we have some (induced) norm $\|\mathbf{G}\| = q$; then for any iteration $k$ we have:
$$\begin{align*}
\|\mathbf{e}^{(k)}\| & = \lVert\mathbf{G}^{k}\;\mathbf{e}^{(0)}\rVert\\
& \le\; \|\mathbf{G}^{k}\|\,\|\mathbf{e}^{(0)}\|\\
& \le\; q^{k}\,\|\mathbf{e}^{(0)}\|\\
\end{align*}
$$
To reach a desired accuracy $\epsilon$ at iteration $k$, we can bound the error as:
$$
\|\mathbf{e}^{(k)}\| \leq q^{k}\,\|\mathbf{e}^{(0)}\| \leq \epsilon
$$
which implies:
$$
\boxed{
   k\geq\frac{\ln(\epsilon/\|\mathbf{e}^{(0)}\|)}{|\ln(\lVert\mathbf{G}\rVert)|}
\quad\blacksquare}
$$

___

## Specific Methods
There are several specific iterative methods that can be used to solve systems of linear equations.

* __Jacobi Method__: The Jacobi method is an iterative solver that, starting from an initial guess, repeatedly refines each unknown in parallel using the residual between the right-hand side and the contributions from other variables. It is easy to implement and parallelize, and converges when the system matrix satisfies appropriate conditions (e.g., strict diagonal dominance). [Let's check out the algorithm.](CHEME-5800-L6c-Algorithm-JacobiMethod-Fall-2026.ipynb)

* __Gauss–Seidel Method__: The Gauss–Seidel method is an iterative solver that, starting from an initial guess, refines each unknown one at a time using the most recent updates, immediately incorporating new values as they become available. This approach typically yields faster convergence than the Jacobi method under the same matrix conditions. It is simple to implement but inherently sequential, and converges when the system matrix is, for example, strictly diagonally dominant. [Let's check out the algorithm.](CHEME-5800-L6c-Algorithm-GaussSeidel-Fall-2026.ipynb)

* __Successive Over-Relaxation (SOR) Method__: The SOR method builds on Gauss–Seidel by introducing a relaxation factor $\omega\in(0,2)$ that _over-relaxes_ each update by blending the new Gauss–Seidel value with the previous iterate to further accelerate convergence. With a well-chosen $\omega$, SOR can dramatically improve the solution speed for diagonally-dominant systems, with convergence guaranteed for $0<\omega<2$. [Let's check out the algorithm.](CHEME-5800-L6c-Algorithm-SOR-Fall-2026.ipynb)

Let's look at an example of each of these methods in action!

> __Example__
>
> [▶ Fun with Iterative Methods](CHEME-5800-L6c-Example-FunWithIterativeSolvers-Fall-2026.ipynb). In this example, we will explore the implementation of various iterative methods for solving (square) systems of linear equations. We will compare the performance of these methods on randomly generated matrices and analyze their convergence behavior.

___

## Lab
In [L6d](../L6d/CHEME-5800-L6d-Lab-IterativeLinearSolvers-Fall-2026.ipynb), we will compare Jacobi, Gauss–Seidel, and SOR on a diagonally dominant linear system using residual histories, iteration counts, and a direct solution as a reference.

## Summary
In this lecture, we explored iterative methods for solving linear systems, focusing on the general algorithm, convergence conditions, and specific implementations.

> __Key Takeaways:__
>
> - **General Algorithm**: Iterative methods start with an initial guess and update the solution using residuals, checking for convergence based on tolerance or iteration limits.
> - **Convergence Criteria**: Methods converge when the spectral radius of the iteration matrix is less than 1, which is related to diagonal dominance of the system matrix.
> - **Specific Methods**: Jacobi, Gauss-Seidel, and SOR differ in how they compute updates, with SOR using a relaxation factor for potentially faster convergence.

___